In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve

In [ ]:
# =====================
# Load & Prepare Dataset
# =====================
df = pd.read_csv("cleaned_behavior_features.csv")
df['converted'] = (df['event_name'] == 'purchase').astype(int)

# Identifiers
user_ids = df['user_pseudo_id']
item_ids = df['item_id']

# Frequency features
df['user_activity_count'] = df['user_pseudo_id'].map(df['user_pseudo_id'].value_counts())
df['item_popularity'] = df['item_id'].map(df['item_id'].value_counts())

# Drop unused
df.drop(columns=['user_pseudo_id','item_id','item_name','event_name',
                 'discounted_price','event_date','region','city','country'],
        inplace=True)

features = [
    'original_price','discount_percent','item_category','campaign_type','channel',
    'hour_of_day','day_of_week','days_since_first_event',
    'user_product_view_count','user_product_purchase_count','user_product_interaction_count',
    'user_activity_count','item_popularity'
]
target = 'converted'
X, y = df[features], df[target]

categorical_cols = ['item_category','campaign_type','channel','day_of_week']
numerical_cols = [col for col in features if col not in categorical_cols]

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

In [ ]:
# =====================
# Train-Test Split
# =====================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

In [ ]:
# =====================
# Helper: Fit model + tune threshold
# =====================
def evaluate_model(model, name):
    # Probabilities
    y_proba = model.predict_proba(X_test)[:,1]

    # Optimal threshold via F1
    precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-6)
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]

    # Predictions
    y_pred_custom = (y_proba >= best_threshold).astype(int)

    # Metrics
    report = classification_report(y_test, y_pred_custom, output_dict=True)
    auc = roc_auc_score(y_test, y_proba)

    return {
        "Model": name,
        "Precision (Buyers)": round(report['1']['precision'],3),
        "Recall (Buyers)": round(report['1']['recall'],3),
        "F1 (Buyers)": round(report['1']['f1-score'],3),
        "ROC-AUC": round(auc,3),
        "Accuracy": round(report['accuracy'],3),
        "Best Threshold": round(best_threshold,3)
    }

In [ ]:
# =====================
# 1️⃣ Random Forest (baseline)
# =====================
rf_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(
        n_estimators=200, random_state=42, class_weight="balanced"
    ))
])
rf_pipeline.fit(X_train, y_train)
rf_results = evaluate_model(rf_pipeline, "Random Forest")

In [ ]:
# =====================
# 2️⃣ MLP (baseline)
# =====================
mlp_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', MLPClassifier(hidden_layer_sizes=(64,32),
                                 activation="relu", max_iter=300, random_state=42))
])
mlp_pipeline.fit(X_train, y_train)
mlp_results = evaluate_model(mlp_pipeline, "MLP")

In [ ]:
from sklearn.linear_model import LogisticRegression
lr_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])
lr_pipeline.fit(X_train, y_train)
lr_results = evaluate_model(lr_pipeline, "Logistic Regression")

In [ ]:
# =====================
# 3️⃣ XGBoost (tuned)
# =====================
xgb_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', XGBClassifier(
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    ))
])

param_grid = {
    'classifier__max_depth': [3,4,5],
    'classifier__min_child_weight': [1,5,10],
    'classifier__gamma': [0,1,2],
    'classifier__subsample': [0.6,0.8,1.0],
    'classifier__colsample_bytree': [0.6,0.8,1.0],
    'classifier__n_estimators': [200,300],
    'classifier__learning_rate': [0.05,0.1,0.2],
    'classifier__scale_pos_weight': [(y==0).sum()/(y==1).sum()]
}

search = RandomizedSearchCV(
    xgb_pipeline,
    param_distributions=param_grid,
    n_iter=15,
    scoring='f1',  # focus on F1
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)
search.fit(X_train, y_train)

xgb_best = search.best_estimator_
xgb_results = evaluate_model(xgb_best, "XGBoost (Tuned)")

Fitting 3 folds for each of 15 candidates, totalling 45 fits


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [12:16:14] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [ ]:
# =====================
# 📊 Compare
# =====================
results_df = pd.DataFrame([rf_results, mlp_results,lr_results, xgb_results])
print(results_df)

                 Model  Precision (Buyers)  Recall (Buyers)  F1 (Buyers)  \
0        Random Forest               0.188            0.431        0.262   
1                  MLP               0.113            0.406        0.177   
2  Logistic Regression               0.203            0.502        0.289   
3      XGBoost (Tuned)               0.193            0.474        0.274   

   ROC-AUC  Accuracy  Best Threshold  
0    0.654     0.805           0.185  
1    0.583     0.697           0.080  
2    0.669     0.802           0.605  
3    0.662     0.799           0.591  


In [ ]:
# Add back discount feature for grouping
discount_idx = all_feature_names.index("discount_percent")
discount_shap = shap_values[:, discount_idx]

# Bucket discounts
discount_buckets = pd.cut(X_test["discount_percent"], bins=[0,20,40,60,80,100])
bucket_summary = pd.DataFrame({
    "bucket": discount_buckets,
    "avg_discount": X_test["discount_percent"],
    "avg_shap_impact": discount_shap
}).groupby("bucket").mean()

print("\n💡 SHAP Impact by Discount Bucket:")
print(bucket_summary)
